# Week 4: Transformer Network

**Learning Objectives:**

- Create positional encodings to capture sequential relationships in data
- Calculate scaled dot-product self-attention with word embeddings
- Implement masked multi-head attention
- Build and train a Transformer model
- Fine-tune a pre-trained transformer model for Named Entity Recognition
- Fine-tune a pre-trained transformer model for Question Answering
- Implement a QA model in TensorFlow and PyTorch
- Fine-tune a pre-trained transformer model to a custom dataset
- Perform extractive Question Answering

---

## **Table of Contents**

- [Transformer Network Intuition](#transformer-network-intuition)
- [Self-Attention](#self-attention)
- [Multi-Head Attention](#multi-head-attention)
- [Transformer Network](#transformer-network)

---

## **Transformer Network Intuition**

This section serves as a high-level introduction to the **Transformer Network**, the architecture that fundamentally shifted NLP away from sequential processing (RNNs) toward parallel, attention-based processing. It sets the stage for a deep dive into self-attention and multi-head attention.

### **High-level Summary**

For years, we relied on RNNs, which acted like a person reading a book one word at a time, moving their finger across the page. While LSTMs made that person better at remembering what happened in the first chapter, they still couldn't skip ahead; they were stuck in a linear, sequential loop.

The Transformer changes the metaphor. Instead of a single finger moving across the text, the Transformer acts like a **camera taking a snapshot** of the entire page at once. By combining the "focus" of **Attention** with the "speed" of **Parallel Processing** (similar to how a CNN looks at all pixels in an image simultaneously), the Transformer can build much richer representations of language ($A_1$ through $A_5$) without waiting for output of each block of the the sequence model every time step.

As we move forward, the focus will shift from *how* we pass information over time to *how* we use **Self-Attention** to let words "talk" to each other across the entire sentence all at once. Introduced in the paper [*"Attention is All You Need"*](https://arxiv.org/abs/1706.03762) by Vaswani et al., this architecture has become the foundation for almost all modern state-of-the-art NLP algorithms.

### **Key Takeaways**

* **The Evolution of Complexity:** 
    * Models have evolved from basic **RNNs** to **GRUs** and **LSTMs**. 
    * While gates (in GRUs/LSTMs) solved vanishing gradient issues and captured long-range dependencies, they increased computational complexity.
* **The Sequential Bottleneck:**
    * RNN-based models are inherently sequential; they process one token at a time from left to right (or both direction but still sequential).
    * This creates a bottleneck because the $n$-th word cannot be processed until the $n-1$ words before it are complete, limiting the ability to use modern parallel hardware (GPUs) effectively.
* **The Transformer Innovation:**
    * The Transformer replaces sequential processing with a **CNN-style parallel approach**.
    * It ingests the entire sequence simultaneously, allowing for massive parallelization and faster training.
* **Core Components:**
    * **Self-Attention:** A mechanism to compute representations for every word in a sentence in parallel by looking at the other words in the same sentence.
    * **Multi-Head Attention:** A "for-loop" over the self-attention process that generates multiple, rich versions of these representations.

---

## **Self-Attention**

This section explains **Self-Attention**, the engine inside the Transformer network that allows it to move beyond fixed word embeddings to dynamic, context-aware representations. Instead of treating a word token as a static vector, the model "looks" at surrounding words to understand exactly how the word is being used in a specific sentence.

### **High-level Summary**

Let's imagine the model is trying to understand the word "l'Afrique" in the sentence *"Jane visite l'Afrique en septembre."* 

The word "l'Afrique" sends out a **Query** ($q^{\langle 3 \rangle}$) asking for context. It scans the **Keys** ($k$) of the other words. It finds that the Key for "visite" ($k^{\langle 2 \rangle}$) is a strong match for its query. This results in a high attention weight for "visite." Consequently, the final representation for "l'Afrique" ($A^{\langle 3 \rangle}$) will absorb the **Value** ($v^{\langle 2 \rangle}$) of the word "visits," effectively encoding the idea: *"This is the 'l'Afrique' that is currently being visited."*

Mathematically, this is a **content-based retrieval system**. In an RNN, information has to travel through a "chain," losing strength over time. In Self-Attention, the distance between the first word and the last word is always **exactly one operation**. This is why Transformers are so much better at "remembering" the beginning of a long document than LSTMs ever were.

<figure style='text-align: center;'>
<img src='assets/self_attention.png' alt='Self Attention' style='width:850px'>
</figure>

### **Key Takeaways**

* **Contextualized Representations ($A^{\langle i \rangle}$):**
    * In standard word embeddings, a word always has the same vector.
    * In the Transformer architecture, self-attention allows every word in a sequence to interact with every other word to determine which parts of the input are most important for its own representation. For example, $A^{\langle 3 \rangle}$ (l'Afrique) will change depending on whether the sentence is about a "visit", "history" or "geography."

* **The Transformation: Q, K, and V:** We start with the input matrix $X \in \mathbb{R}^{n \times d_{model}}$, where each row is a word embedding of size $d$. For every word, we learn three separate linear transformations. We multiply the input matrix $X$ by three weight matrices ($W^Q, W^K, W^V$) that are updated during training:

    * **Queries ($Q = XW^Q$):** What I am looking for.
    * **Keys ($K = XW^K$):** What I contain (to match against queries).
    * **Values ($V = XW^V$):** The information I actually contribute.

    $$Q \in \mathbb{R}^{n \times d_k}, \quad K \in \mathbb{R}^{n \times d_k}, \quad V \in \mathbb{R}^{n \times d_v}$$

* **Calculating the Attention Scores:** We measure the compatibility between a Query and a Key using the dot product. This tells us how much "attention" word $i$ should pay to word $j$.

    $$Score = QK^\top$$

    The resulting matrix is of size $n \times n$ (sequence length by sequence length), where the value at $(i, j)$ is the raw attention score between the $i$-th and $j$-th words.

* **Scaling and Softmax:** If the dimensionality $d_k$ is large, the dot products can grow very large, pushing the Softmax function into regions where gradients are extremely small (the "vanishing gradient" problem). To fix this, we scale by $\sqrt{d_k}$. We then apply a **Softmax** along each row to ensure the attention weights are positive and sum to 1:

    $$\text{Weights} = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)$$

* **The Final Output (hidden states):** Finally, we multiply the weights by the **Value** matrix $V$. This step "extracts" the relevant information. If word $i$ has a high weight for word $j$, the resulting representation for word $i$ will be heavily influenced by the features of word $j$.

    $$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

<figure style='text-align: center;'>
<img src='assets/attention_is_all_you_need_step1_2.png' alt='Self Attention Matrix Form' style='width:950px'>
<figcaption>From <a href='https://newsletter.theaiedge.io'>The AiEdge.io.</a></figcaption>
</figure>

* **Parallel Processing:**
    * Unlike RNNs, which process word by word, these computations are done for all words in the sentence simultaneously using matrix operations, which is ideal for GPU to process.

---

## **Multi-Head Attention**

This section explains **Multi-Head Attention**, which is essentially a way to run the self-attention mechanism multiple times in parallel. By using different sets of learned weights, the model can ask and answer multiple "questions" about the same word simultaneously.

### **High-level Summary**

In self-attention a single person trying to understand a sentence by asking one question, whereas in Multi-Head Attention a **panel of experts**, each looking for something different.

When processing the word "l'Afrique," one expert (Head) might notice it is a destination, while another expert notices that the trip is happening in September. This is achieved by ensuring that each set of weights ($W_i^Q, W_i^K, W_i^V$) learns a different subspace. By the time the model finishes this parallel processing, the representation of "l'Afrique" is a complex structure that knows it is being *visited* by *Jane* in *September*.

This parallel "committee" of attention heads is what gives Transformers their incredible expressive power. Because these heads don't rely on each other, the model can leverage modern hardware to calculate all these relationships at once, avoiding the slow, sequential "left-to-right" crawl of an RNN.

### **Key Takeaways**

* **The "For-Loop" Intuition:**
    * Multi-head attention is conceptually like running a "for-loop" over the self-attention process. Each iteration $i$ is called a **Head**.
    * While we think of it as a loop, in practice, all heads are computed in **parallel** because their calculations are independent.
* **Multiple Questions, Multiple Answers:**
    * Each head $i$ uses its own set of learned weight matrices ($W_i^Q, W_i^K, W_i^V$).
    * For a given head $i$, the Query, Key, and Value matrices are calculated as:
      $$Q_i = XW_i^Q, \quad K_i = XW_i^K, \quad V_i = XW_i^V$$
    * This allows the model to look for different types of relationships at the same time:
        * **Head 1:** Might ask "What is happening?" (focusing on verbs).
        * **Head 2:** Might ask "When is it happening?" (focusing on time).

    >**Note:** In Multi-Head Self-Attention, the inputs are simplified to $Q=K=V=x$. We don't pre-calculate these vectors because each head $i$ already contains its own learned weight matrices ($W_i^Q, W_i^K, W_i^V$) to perform those transformations internally. We maintain the distinct $Q, K$, and $V$ notation in diagrams for two reasons:
    >* **Avoid Redundancy:** Applying transformations before the multi-head block and then again inside the heads would be mathematically redundant.
    >* **Encoder-Decoder Attention:** In the part of the Transformer where the input and output interact, $Q, K$, and $V$ are **not** the same; $Q$ comes from the decoder, while $K$ and $V$ come from the encoder.

* **Concatenation and Projection:**
    * Each head produces its own output: 
      $$\text{head}_i = \text{Attention}(Q_i, K_i, V_i) = \text{softmax}\left(\frac{Q_i K_i^\top}{\sqrt{d_k}}\right)V_i$$
    * All $H$ heads are **concatenated** together and then multiplied by an additional weight matrix $W^O$ to project the result back into the model's standard dimension:
      $$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h)W^O$$

<figure style='text-align: center'>
<img src='assets/multi_head_attention.png' alt='Multi-Head Attention' style='width: 950px'>
</figure>

Now, we have the overall picture of the input sequence represenation. Here is the schematic of three steps to generate the hidden features of the input sequence using multi-head attention mechanism:

<figure style='text-align: center'>
<img src='assets/attention_is_all_you_need.png' alt='Multi-Head Attention' style='width: 950px'>
<figcaption>From <a href='https://newsletter.theaiedge.io'>The AiEdge.io.</a></figcaption>
</figure>


---

## **Transformer Network**

The Transformer architecture represents a shift from sequential processing to a highly parallelized, attention-driven system. It consists of two main parts: the **Encoder**, which understands the source language, and the **Decoder**, which generates the target language.

### **High-level Summary**

The Transformer works like a highly sophisticated coordination between a **Reader (Encoder)** and a **Writer (Decoder)**. Instead of reading left-to-right, the Reader takes a "snapshot" of the whole French sentence. To ensure it knows that "Jane" comes before "Africa," it uses **Positional Encoding**, adding a mathematical "timestamp" to each word.

As the Writer starts, it asks: "Given I've written 'Jane visits', what should I look for in the French summary?" This is where the **Encoder-Decoder Attention** shines. The Decoder sends out a **Query**, and the Encoder provides the relevant **Keys** and **Values**. 


### **Key Takeaways**

* **The Encoder Block ($N$ layers):**
    * Takes the input sentence (e.g., French) and passes it through a **Multi-Head Attention (MHA)** layer and a **Feed-Forward Neural Network (FFNN)**.
    * In the original paper, this block is repeated **$n=6$ times** to create a very rich feature representation of the source sentence.
* **The Decoder Block ($N$ layers):**
    * Accepts the target sequence (e.g., English) generated so far.
    * Uses two types of attention:
        1.  **Masked Multi-Head Attention:** Processes the output words generated so far.
        2.  **Encoder-Decoder Attention:** Uses the Decoder as the "Query" ($Q$) to pull information from the Encoder’s "Keys" ($K$) and "Values" ($V$).
* **Positional Encoding:**
    * Since Transformers process words in parallel, they have no inherent sense of word order. 
    * To understand how the model differentiates positions, it uses a set of sinusoids with different frequencies. For a position $pos$ and dimension $i$:
        
        $$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
        $$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
        
        This ensures that every position has a unique signature that the model can learn to interpret as "distance" between words. To understand why these functions are chosen for this purpose, read [Positional Encoding in Transformers](https://medium.com/thedeephub/positional-encoding-explained-a-deep-dive-into-transformer-pe-65cfe8cfe10b).

* **Training with Masking:**
    * During training, the model has the full correct translation. **Masking** hides future words, forcing the model to predict the next word based only on previous ones.
* **Architectural "Bells and Whistles":**
    * **Residual Connections:** Pass information (like position) directly through layers to prevent signal loss.
    * **Add & Norm:** A normalization step similar to Batch Norm that stabilizes and speeds up training. For details, read [Layer Normalization in Transformer](https://medium.com/@sachinsoni600517/layer-normalization-in-transformer-1a2efbff8b85).
    * **Dropout Regularization:** We regularize the model within both the **MHA** block and the **FFNN** block to prevent overfitting and ensure robust feature learning.
        1. **Within the MHA Block** Dropout is strategically placed at two stages:
            * **Attention Weight Dropout:** Applied immediately after the Softmax** operation but before multiplying by the Value ($V$) matrix. It prevents the model from over-relying on specific "keys" or token relationships, forcing it to distribute its attention across the sequence.
            
            $$\text{Dropout}(\text{Softmax}(\frac{QK^\top}{\sqrt{d_k}}))V$$
            
            * **Output Projection Dropout:** Applied after the attention heads are concatenated and passed through the Output Projection (Linear layer), but before the residual addition. It regularizes the contextualized representation before it is merged with the identity connection.
            
            $$\text{LayerNorm}(x + \text{Dropout}(\text{MHA\_Output}))$$

        2. **Within the FFNN** In the FFNN block, applied after the final linear transformation of the FFNN, just before the residual addition and Layer Normalization. This prevents the high-dimensional internal activations of the FFNN from memorizing training patterns, ensuring the "processed" information generalizes well to new data.
            
            $$\text{LayerNorm}(x + \text{Dropout}(\text{FFNN\_Output}))$$

    * **Linear & Softmax:** The final layers that convert the decoder's output into a probability distribution over the entire vocabulary.

<figure style='text-align: center'>
<img src='assets/transformer_details.png' alt='Transformer Architecture' style='width: 950px'>
</figure>


### **Links to Bonus Learning**

* [A Mathematical Framework for Transformer Circuits](https://transformer-circuits.pub/2021/framework/index.html)
* [Transformer Architecture - Part I](https://pub.towardsai.net/transformer-architecture-part-1-d157b54315e6)
* [Transformer Architecture - Part II]()
* [Layer Normalization in Transformer](https://medium.com/@sachinsoni600517/layer-normalization-in-transformer-1a2efbff8b85)
* [Positional Encoding in Transformers](https://medium.com/thedeephub/positional-encoding-explained-a-deep-dive-into-transformer-pe-65cfe8cfe10b)
* [Nice video to build intuition about Self-Attention in transformers](https://www.youtube.com/watch?v=eMlx5fFNoYc&list=PLZHQObOWTQDNU6R1_67000Dx_ZCJB-3pi&index=7)
* [Nice Video about MLP in transformers](https://www.youtube.com/watch?v=9-Jl0dxWQs8&list=PLZHQObOWTQDNU6R1_67000Dx_ZCJB-3pi&index=8)